
# 01 - Loading and merging the SAMI DR3 tables

**Goal:** combine the SAMI tables we need for the Faber-Jackson relation into one table, with one row per galaxy.

The Faber-Jackson relation links a galaxy's luminosity (or stellar mass) to its stellar velocity dispersion (sigma). To study it we need, for each galaxy:

| What we need | Table | Column(s) |
|---|---|---|
| Luminosity | input catalogues (GAMA + Clusters) | `m_r` (absolute r-band magnitude) |
| Stellar mass | input catalogues | `mstar` (log10 of M*/Msun) |
| Velocity dispersion | `samiDR3StelKin` | `sigma_re`, `sigma_re_err` |
| Morphology | `samiDR3VisualMorphology` | `type` |

SAMI observed galaxies in two parts of the sky: the GAMA regions (field and group galaxies) and 8 galaxy clusters. These are in two separate input catalogues, so we load both and stack them together.

Keeping both is useful because galaxies are not spread randomly through space. Redshift surveys show they trace a "cosmic web" of filaments and walls around empty voids, with clusters sitting at the dense nodes where filaments meet. So the GAMA and Cluster galaxies live in very different environments, which lets us test later whether environment affects the Faber-Jackson relation.

All tables share the SAMI galaxy ID `catid`, which we use to join them.

In [1]:
import pandas as pd
import numpy as np

## 1. Load the raw tables

In [2]:
gama = pd.read_csv('../data/InputCatGAMADR3.csv', index_col=0)
clusters = pd.read_csv('../data/samiDR3InputCatClusters.csv', index_col=0)
kin = pd.read_csv('../data/samiDR3StelKin.csv', index_col=0)
morph = pd.read_csv('../data/samiDR3VisualMorphology.csv', index_col=0)

print('GAMA input catalogue:   ', len(gama), 'galaxies')
print('Cluster input catalogue:', len(clusters), 'galaxies')
print('Stellar kinematics:     ', len(kin), 'rows')
print('Visual morphology:      ', len(morph), 'galaxies')

GAMA input catalogue:    5536 galaxies
Cluster input catalogue: 1433 galaxies
Stellar kinematics:      3426 rows
Visual morphology:       3068 galaxies


In [3]:
kin[['catid', 'cubeidpub', 'sigma_re', 'sigma_re_err', 'lambdar_re', 'aper_corr_flag']].head()

,catid,cubeidpub,sigma_re,sigma_re_err,lambdar_re,aper_corr_flag
0,6821,6821_A,NaN,NaN,NaN,NaN
1,6837,6837_A,NaN,NaN,NaN,NaN
2,7139,7139_A,235.61281,3.014763,NaN,0.0
3,7206,7206_A,106.10177,2.071345,0.549376,0.0
4,7289,7289_A,205.14528,1.877292,0.071932,0.0


## 2. Combine the GAMA and Cluster input catalogues

First check that no galaxy appears in both catalogues, so we don't double count. The two catalogues have slightly different columns, so we only keep the columns we need.

In [4]:
overlap = set(gama['catid']) & set(clusters['catid'])
print('Galaxies in both GAMA and Cluster catalogues:', len(overlap))

Galaxies in both GAMA and Cluster catalogues: 0


In [5]:
cols = ['catid', 'ra_obj', 'dec_obj', 'z_spec', 'm_r', 'r_e', 'ellip', 'mstar', 'g_i', 'bad_class']

gama_small = gama[cols].copy()
gama_small['environment'] = 'GAMA'
gama_small['is_mem'] = np.nan          

clusters_small = clusters[cols + ['is_mem']].copy()
clusters_small['environment'] = 'Cluster'

parent = pd.concat([gama_small, clusters_small], ignore_index=True)
print('Combined input catalogue:', len(parent), 'galaxies')
parent['environment'].value_counts()

Combined input catalogue: 6969 galaxies


environment
GAMA       5536
Cluster    1433
Name: count, dtype: int64

## 3. Deal with repeat observations in the kinematics table

Some galaxies were observed more than once, so they have more than one row (cube) in the kinematics table. We want one row per galaxy.

In [6]:
n_repeats = kin['catid'].duplicated().sum()
print('Number of repeat rows:', n_repeats)

repeats = kin[kin['catid'].duplicated(keep=False)]
repeats[['catid', 'cubeidpub', 'sigma_re', 'sigma_re_err']].sort_values('catid').head(6)

Number of repeat rows: 358


,catid,cubeidpub,sigma_re,sigma_re_err
37,15165,15165_A,180.070000,2.136608
38,15165,15165_B,NaN,NaN
40,15220,15220_A,45.762318,7.479450
41,15220,15220_B,NaN,NaN
61,16863,16863_B,NaN,NaN
62,16863,16863_A,NaN,NaN


Often only one of the repeat cubes has a measured sigma. Our rule: for each galaxy keep the cube with the smallest `sigma_re_err` (the most precise measurement). Sorting puts the smallest error first and pushes missing values (NaN) to the end, so keeping the first row per galaxy does what we want.

In [7]:
kin_best = kin.sort_values('sigma_re_err').drop_duplicates('catid', keep='first')
print('Kinematics rows after removing repeats:', len(kin_best))

Kinematics rows after removing repeats: 3068


## 4. Merge everything on `catid`

We use a left join onto the kinematics table, so we keep every galaxy SAMI has kinematics for (even if sigma is missing). That way the selection notebook can count how many galaxies each cut removes.

In [8]:
kin_cols = ['catid', 'cubeidpub', 'sigma_re', 'sigma_re_err', 'sigma_3kpc_round', 'sigma_3kpc_round_err',
            'aper_corr_flag', 'lambdar_re', 'lambdar_re_err', 'vsigma_re']

merged = kin_best[kin_cols].merge(parent, on='catid', how='left')
merged = merged.merge(morph, on='catid', how='left')

print('Merged table:', len(merged), 'galaxies')
print('Missing input catalogue info:', merged['environment'].isna().sum())
print('Missing morphology:          ', merged['type'].isna().sum())
merged.head()

Merged table: 3068 galaxies
Missing input catalogue info: 72
Missing morphology:           0


,catid,cubeidpub,sigma_re,sigma_re_err,sigma_3kpc_round,sigma_3kpc_round_err,aper_corr_flag,lambdar_re,lambdar_re_err,vsigma_re,...,z_spec,m_r,r_e,ellip,mstar,g_i,bad_class,environment,is_mem,type
0,321059,321059_A,86.169044,0.544646,87.018580,0.679822,0.0,0.483175,0.002529,0.595053,...,0.005820,-18.860000,6.980000,0.62960,9.670000,1.100000,0.0,GAMA,NaN,1.0
1,423267,423267_A,103.233350,0.570141,103.960690,0.704756,0.0,0.323014,0.003646,0.377509,...,0.013000,-19.870000,4.840000,0.47810,10.020000,1.050000,0.0,GAMA,NaN,1.0
2,78827,78827_A,95.779580,0.628522,97.138626,0.654244,0.0,0.246275,0.004738,0.251419,...,0.029260,-20.560000,3.370000,0.21830,10.300000,1.070000,0.0,GAMA,NaN,1.5
3,289113,289113_A,123.384960,0.642291,120.216140,0.732519,0.0,0.363310,0.003444,0.413235,...,0.023870,-20.890000,6.510000,0.43970,10.520000,1.180000,0.0,GAMA,NaN,1.0
4,9403801002,9403801002_A,115.781456,0.647606,118.395650,0.725362,0.0,0.326045,0.004923,0.318053,...,0.026791,-20.439213,4.399992,0.24108,10.361357,1.198622,0.0,Cluster,1.0,0.0


## 5. Save the merged table for the next notebook

In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)
merged.to_csv('../data/processed/sami_merged.csv', index=False)
print('Saved', len(merged), 'rows to data/processed/sami_merged.csv')

Saved 3068 rows to data/processed/sami_merged.csv
